# SunSight Training `YOLO11s-Seg` — Versi KAGGLE GPU (Anti-Putus Internet!)

Kenapa pakai **Kaggle `Save & Run All`**?
- Cukup klik **Save Version → Save & Run All (Commit)** dengan Accelerator **GPU T4 x2** atau **GPU P100**.
- Setelah klik, **kamu boleh tutup browser / matikan laptop dan tidur!** Training berjalan penuh di server Kaggle tanpa pernah terputus karena Wi-Fi.
- Besok pagi tinggal buka tab **Output** dan unduh **`best.onnx`** (~38 MB) + angka tabel jurnal (`journal_metrics.json`).

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime opencv-python-headless

In [ ]:
# 1. Temukan dataset otomatis dari /kaggle/input/ (upload sunsight_yolo11_seg.zip sebagai Kaggle Dataset)
import shutil, yaml
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_DATA = Path('/kaggle/working/sunsight_dataset')

yamls = list(INPUT_ROOT.rglob('dataset.yaml'))
if not yamls:
    raise FileNotFoundError('Tambahkan Kaggle Dataset yang berisi sunsight_yolo11_seg (dataset.yaml) lewat tombol Add Input!')

SRC_DATASET = yamls[0].parent
print('Dataset ditemukan di:', SRC_DATASET)

if WORK_DATA.exists():
    shutil.rmtree(WORK_DATA)
shutil.copytree(SRC_DATASET, WORK_DATA)

# Pastikan path di dataset.yaml menunjuk ke folder writable /kaggle/working/sunsight_dataset
cfg = yaml.safe_load((WORK_DATA / 'dataset.yaml').read_text(encoding='utf-8'))
cfg['path'] = str(WORK_DATA)
(WORK_DATA / 'dataset.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Dataset siap di:', WORK_DATA)

In [ ]:
# 2. Training Model Utama: YOLO11s-Seg pada Esri Kota Bogor + Hard Negatives + Anti-Color-Bias + Anti-Merged Dense Roofs
import json
from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')
model.train(
    data=str(WORK_DATA / 'dataset.yaml'),
    epochs=150,
    patience=30,
    imgsz=1024,
    batch=8,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    overlap_mask=False,
    hsv_h=0.04,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=180.0,
    flipud=0.5,
    fliplr=0.5,
    scale=0.35,
    mosaic=1.0,
    close_mosaic=20,
    project='/kaggle/working/runs',
    name='yolo11s_seg_esri_bogor',
    exist_ok=True,
    plots=True,
)

best_pt = Path('/kaggle/working/runs/yolo11s_seg_esri_bogor/weights/best.pt')
best_model = YOLO(str(best_pt))
# Evaluasi standar COCO/Ultralytics (conf default 0.001 agar kurva PR & mAP@50 tidak terpotong!)
m_rgb = best_model.val(data=str(WORK_DATA / 'dataset.yaml'), split='test', imgsz=1280, max_det=300)

# Export langsung ke /kaggle/working/best.onnx agar mudah diunduh dari tab Output Kaggle
onnx_path = Path(best_model.export(format='onnx', imgsz=1280, simplify=True, opset=17))
shutil.copy2(onnx_path, '/kaggle/working/best.onnx')
shutil.copy2(best_pt, '/kaggle/working/best.pt')

summary = {
    'esri_rgb_yolo11s_seg': {
        'mask_precision': round(float(m_rgb.seg.mp), 4),
        'mask_recall': round(float(m_rgb.seg.mr), 4),
        'mask_map50': round(float(m_rgb.seg.map50), 4),
        'mask_map50_95': round(float(m_rgb.seg.map), 4),
    }
}
print('=== HASIL TEST SET (RGB) ===', json.dumps(summary, indent=2))

In [ ]:
# 3. (Otomatis untuk Tabel 1 Jurnal SINTA 4) Eksperimen Versi CLAHE LAB
import cv2
from tqdm import tqdm

RUN_CLAHE_ALSO = True  # Set False jika ingin selesai 2x lebih cepat (hanya 1 model utama)

def apply_clahe_lab(bgr_img):
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge((clahe.apply(l), a, b)), cv2.COLOR_LAB2BGR)

if RUN_CLAHE_ALSO:
    CLAHE_DATA = Path('/kaggle/working/sunsight_dataset_clahe')
    if CLAHE_DATA.exists():
        shutil.rmtree(CLAHE_DATA)
    shutil.copytree(WORK_DATA, CLAHE_DATA)
    for split in ('train', 'val', 'test'):
        for img_p in tqdm(list((CLAHE_DATA / 'images' / split).glob('*.*')), desc=f'CLAHE {split}'):
            bgr = cv2.imread(str(img_p))
            if bgr is not None:
                cv2.imwrite(str(img_p), apply_clahe_lab(bgr), [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    cfg_c = yaml.safe_load((CLAHE_DATA / 'dataset.yaml').read_text(encoding='utf-8'))
    cfg_c['path'] = str(CLAHE_DATA)
    (CLAHE_DATA / 'dataset.yaml').write_text(yaml.safe_dump(cfg_c, sort_keys=False), encoding='utf-8')

    model_c = YOLO('yolo11s-seg.pt')
    model_c.train(
        data=str(CLAHE_DATA / 'dataset.yaml'),
        epochs=150,
        patience=30,
        imgsz=1024,
        batch=8,
        optimizer='AdamW',
        lr0=0.001,
        cos_lr=True,
        overlap_mask=False,
        hsv_h=0.04,
        hsv_s=0.7,
        hsv_v=0.5,
        degrees=180.0,
        flipud=0.5,
        fliplr=0.5,
        scale=0.35,
        mosaic=1.0,
        close_mosaic=20,
        project='/kaggle/working/runs',
        name='yolo11s_seg_esri_clahe',
        exist_ok=True,
        plots=True,
    )
    best_c_pt = Path('/kaggle/working/runs/yolo11s_seg_esri_clahe/weights/best.pt')
    best_c = YOLO(str(best_c_pt))
    m_clahe = best_c.val(data=str(CLAHE_DATA / 'dataset.yaml'), split='test', imgsz=1280, max_det=300)
    onnx_c = Path(best_c.export(format='onnx', imgsz=1280, simplify=True, opset=17))
    shutil.copy2(onnx_c, '/kaggle/working/best_clahe.onnx')
    summary['esri_clahe_yolo11s_seg'] = {
        'mask_precision': round(float(m_clahe.seg.mp), 4),
        'mask_recall': round(float(m_clahe.seg.mr), 4),
        'mask_map50': round(float(m_clahe.seg.map50), 4),
        'mask_map50_95': round(float(m_clahe.seg.map), 4),
    }

Path('/kaggle/working/journal_metrics.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('=== SELESAI! RINGKASAN JURNAL ===')
print(json.dumps(summary, indent=2))